In [1]:
import os, sys, json
import pandas as pd
from google.cloud import bigquery
PROJECT_ID = "project-5eb4889f-74fb-4ad5-a76"
LOCATION   = "EU"

creds = None

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
elif os.environ.get("GCP_SA_KEY"):
    from google.oauth2 import service_account
    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"])

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)
print("проєкт:", client.project)

проєкт: project-5eb4889f-74fb-4ad5-a76


In [2]:
raw = client.query(f"""
    SELECT ingested_at, business_date, payload
    FROM `{PROJECT_ID}.nbu_raw.raw_rates`
    WHERE business_date = (
        SELECT MAX(business_date) FROM `{PROJECT_ID}.nbu_raw.raw_rates`
    )
""").to_dataframe()

print(raw.shape)
print(raw["business_date"].unique())

(45, 3)
<DateArray>
[datetime.date(2026, 9, 29)]
Length: 1, dtype: dbdate


/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [4]:
parsed = pd.json_normalize(raw["payload"].map(json.loads))
parsed["ingested_at"] = raw["ingested_at"]
parsed["business_date"] = raw["business_date"]

df = parsed.copy()
df["currency_code"] = df["cc"].str.strip().str.upper()
df["rate"] = df["rate"].astype(float)

# зерно таблиці: одна валюта на одну дату, беремо найсвіжіший запис
today_df = (df.sort_values("ingested_at")
              .drop_duplicates(subset=["business_date", "currency_code"], keep="last"))

print(today_df.shape)

(45, 9)


In [5]:
dim_currency = client.query(f"""
    SELECT currency_key, currency_code
    FROM `{PROJECT_ID}.nbu_dwh.dim_currency`
""").to_dataframe()

# на випадок повторного запуску прибираємо старий currency_key
today_df = today_df.drop(columns=["currency_key"], errors="ignore")

today_df = today_df.merge(dim_currency, on="currency_code", how="left")
today_df["currency_key"] = today_df["currency_key"].fillna(-1).astype(int)

today_df["date_key"] = (pd.to_datetime(today_df["business_date"])
                          .dt.strftime("%Y%m%d").astype(int))
today_df["dw_load_ts"] = pd.Timestamp.now(tz="UTC")

today_df = today_df[["date_key", "currency_key", "rate", "business_date", "dw_load_ts"]]

print(today_df.shape)
today_df.head()

(45, 5)


/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,date_key,currency_key,rate,business_date,dw_load_ts
0,20260929,10,0.33508,2026-09-29,2026-09-28 18:41:56.158205+00:00
1,20260929,45,2.72960,2026-09-29,2026-09-28 18:41:56.158205+00:00
2,20260929,33,4.50160,2026-09-29,2026-09-28 18:41:56.158205+00:00
3,20260929,6,53.86650,2026-09-29,2026-09-28 18:41:56.158205+00:00
4,20260929,35,1.33388,2026-09-29,2026-09-28 18:41:56.158205+00:00


In [6]:
fact_schema = [
    bigquery.SchemaField("date_key",       "INTEGER"),
    bigquery.SchemaField("currency_key",   "INTEGER"),
    bigquery.SchemaField("rate",           "FLOAT"),
    bigquery.SchemaField("business_date",  "DATE"),
    bigquery.SchemaField("dw_load_ts",     "TIMESTAMP"),
]

FACT_ID = f"{PROJECT_ID}.nbu_dwh.fact_exchange_rate"

table = bigquery.Table(FACT_ID, schema=fact_schema)
table.time_partitioning = bigquery.TimePartitioning(field="business_date")
table.clustering_fields = ["currency_key"]
client.create_table(table, exists_ok=True)

print("таблиця готова:", FACT_ID)

таблиця готова: project-5eb4889f-74fb-4ad5-a76.nbu_dwh.fact_exchange_rate


In [7]:
STG_ID = f"{PROJECT_ID}.nbu_dwh._stg_rates_today"

cfg = bigquery.LoadJobConfig(schema=fact_schema, write_disposition="WRITE_TRUNCATE")
client.load_table_from_dataframe(today_df, STG_ID, job_config=cfg).result()
print("stg записано:", len(today_df))

merge_sql = f"""
MERGE `{FACT_ID}` AS target
USING `{STG_ID}` AS source
    ON  target.date_key     = source.date_key
    AND target.currency_key = source.currency_key

WHEN MATCHED THEN
    UPDATE SET rate       = source.rate,
               dw_load_ts = source.dw_load_ts

WHEN NOT MATCHED THEN
    INSERT (date_key, currency_key, rate, business_date, dw_load_ts)
    VALUES (source.date_key, source.currency_key, source.rate,
            source.business_date, source.dw_load_ts)
"""

job = client.query(merge_sql)
job.result()
print("MERGE виконано, змінено рядків:", job.num_dml_affected_rows)

/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


stg записано: 45
MERGE виконано, змінено рядків: 45


In [8]:
check_sql = f"""
SELECT
  (SELECT COUNT(*) FROM (
      SELECT date_key, currency_key
      FROM `{FACT_ID}`
      GROUP BY date_key, currency_key
      HAVING COUNT(*) > 1)) AS dup_grain,
  COUNT(DISTINCT business_date) AS uniq_dates,
  COUNT(*) AS total_rows
FROM `{FACT_ID}`
"""
print(client.query(check_sql).to_dataframe())

   dup_grain  uniq_dates  total_rows
0          0           1          45


/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
